# ColPali 文档图像检索

> 中文学习版导读｜原始案例：`multi_model_rag_with_colpali.ipynb`  
> 所属阶段：第七阶段：多模态与结构化数据

## 本节目标

直接从页面图像和版面中检索相关内容。

## 阅读重点

适合扫描件、图表和复杂布局，资源需求高于文本 RAG。

## 运行约定

- 从项目根目录启动 JupyterLab。
- 模型和服务地址统一读取 `config/models.toml`。
- API Key 等敏感信息只写入本地 `.env`。
- 本 Notebook 保留上游实现用于技术核对；新增中文导读负责说明学习顺序、配置方式和实验重点。
- 运行前阅读同目录的 `multi_model_rag_with_colpali.zh-CN.md`。


In [ ]:
# 统一配置入口：模型名和服务地址来自 config/models.toml，密钥来自 .env
from pathlib import Path
import os
import sys

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (_current, *_current.parents)
    if (candidate / "pyproject.toml").exists()
)
_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from rag_techniques_zh.config import apply_runtime_environment
settings = apply_runtime_environment()
CHAT_MODEL = settings.openai.chat_model
EMBEDDING_MODEL = settings.openai.embedding_model
EVALUATION_MODEL = settings.openai.evaluation_model
OPENAI_API_KEY = settings.openai.api_key
OPENAI_BASE_URL = settings.openai.base_url
CONTEXTUAL_BASE_URL = settings.contextual.base_url
COHERE_CHAT_MODEL = settings.cohere.chat_model
COHERE_EMBEDDING_MODEL = settings.cohere.embedding_model
COHERE_RERANK_MODEL = settings.cohere.rerank_model
GOOGLE_CHAT_MODEL = settings.google.chat_model
GROQ_FAST_MODEL = settings.groq.fast_model
GROQ_LARGE_MODEL = settings.groq.large_model
OLLAMA_EMBEDDING_MODEL = settings.ollama.embedding_model
SENTENCE_TRANSFORMER_MODEL = settings.local_models.sentence_transformer_model
CROSS_ENCODER_MODEL = settings.local_models.cross_encoder_model
CONTEXTUAL_RERANK_MODEL = settings.contextual.rerank_model
if settings.cohere.api_key:
    os.environ.setdefault("CO_API_KEY", settings.cohere.api_key)

print("当前模型配置：", {
    "chat": CHAT_MODEL,
    "embedding": EMBEDDING_MODEL,
    "evaluation": EVALUATION_MODEL,
    "base_url": OPENAI_BASE_URL,
})


### Overview:
This code implements one of the multiple ways of multi-model RAG. This project processes a PDF file, retrieves relevant content using Colpali, and generates answers using a multi-modal RAG system. The process includes document indexing, querying, and summarizing with the Gemini model.

### Key Components:
- **RAGMultiModalModel**: Used for document indexing and retrieval.
- **PDF Processing**: Downloads and processes "Attention is All You Need" paper.
- **Gemini Model**: Used for content generation from retrieved images and queries.
- **Base64 Encoding/Decoding**: Manages image data retrieved during search.

### Diagram:
   <img src="../images/multi_model_rag_with_colpali.svg" alt="Reliable-RAG" width="300">

### Motivation:
To enable efficient querying and content generation from multi-modal documents (PDFs with text and images) in response to natural language queries.

### Method Details:
- Indexing: The PDF is indexed using the `RAGMultiModalModel`, storing both text and image data.
- Querying: Natural language queries retrieve relevant document segments.
- Image Processing: Images from the document are decoded, displayed, and used in conjunction with the Gemini model to generate content.

### Benefits:
- Multi-modal support for both text and images.
- Streamlined retrieval and summarization pipeline.
- Flexible content generation using advanced LLMs (Gemini model).

### Implementation:
- PDF is indexed, and the content is split into text and image segments.
- A query is run against the indexed document to fetch the relevant results.
- Retrieved image data is decoded and passed through the Gemini model for answer generation.

### Summary:
This project integrates document indexing, retrieval, and content generation in a multi-modal setting, enabling efficient queries on complex documents like research papers.

## Setup


In [ ]:
# Install required packages
!pip install pillow python-dotenv

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git qwen-vl-utils flash-attn optimum auto-gptq bitsandbytes

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [ ]:
# Install required packages
!pip install base64 byaldi os ragmultimodalmodel

# Package Installation

The cell below installs all necessary packages required to run this notebook. If you're running this notebook in a new environment, execute this cell first to ensure all dependencies are installed.

In [ ]:
# Install required packages
!pip install byaldi

In [ ]:
import base64
import os
settings.require("HF_TOKEN") # to download the ColPali model
from byaldi import RAGMultiModalModel

In [ ]:
RAG = RAGMultiModalModel.from_pretrained("vidore/colpali-v1.2", verbose=1)

### Download the "Attention is all you need" paper

In [ ]:
!wget https://arxiv.org/pdf/1706.03762
!mkdir docs
!mv 1706.03762 docs/attention_is_all_you_need.pdf

### Indexing

In [ ]:
RAG.index(
    input_path="./docs/attention_is_all_you_need.pdf",
    index_name="attention_is_all_you_need",
    store_collection_with_index=True, # set this to false if you don't want to store the base64 representation
    overwrite=True
)

### Query time

In [ ]:
query = "What is the BLEU score of the Transformer (base model)?"

In [ ]:
results = RAG.search(query, k=1)

### Actual image data

In [ ]:
image_bytes = base64.b64decode(results[0].base64)

In [ ]:
filename = 'image.jpg'  # I assume you have a JPG file
with open(filename, 'wb') as f:
  f.write(image_bytes)

In [ ]:
from IPython.display import Image

display(Image(filename))

## Test using gemini-1.5-flash

In [ ]:
import google.generativeai as genai

genai.configure(api_key=settings.google.api_key)
model = genai.GenerativeModel(model_name=GOOGLE_CHAT_MODEL)

In [ ]:
from PIL import Image
image = Image.open(filename)

In [ ]:
response = model.generate_content([image, query])
print(response.text)